In [ ]:
# 1. 구글 드라이브 연동 (실행 후 링크나 팝업창에서 권한 허용 눌러줘!)
from google.colab import drive
import os
import urllib.request

drive.mount('/content/drive')

# 2. 구글 드라이브 내에 졸업프로젝트 전용 데이터 폴더 생성
# 드라이브의 '내 드라이브(MyDrive)' 아래에 'Colab_Data' 폴더가 만들어져.
drive_data_path = '/content/drive/MyDrive/Colab_Data'
os.makedirs(drive_data_path, exist_ok=True)

# 3. CIFAR-10 압축본 다운로드 URL 지정 (fast.ai의 초고속 미러 서버)
cifar_url = 'https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz'
destination_path = os.path.join(drive_data_path, 'cifar10.tgz')

# 4. 다운로드 시작
if not os.path.exists(destination_path):
    print("🚀 구글 드라이브로 CIFAR-10 압축 파일 다운로드를 시작합니다...")
    print("대략 10초~20초 정도 소요됩니다. 잠시만 기다려주세요.")

    urllib.request.urlretrieve(cifar_url, destination_path)

    print("\n✅ 다운로드 완료!")
    print(f"파일 저장 위치: {destination_path}")
    print(f"현재 파일 크기: {os.path.getsize(destination_path) / (1024*1024):.2f} MB")
else:
    print("✨ 이미 구글 드라이브에 cifar10.tgz 파일이 안전하게 존재합니다!")

Mounted at /content/drive
✨ 이미 구글 드라이브에 cifar10.tgz 파일이 안전하게 존재합니다!


In [ ]:
import os
import time
import tarfile
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ 현재 사용 중인 장치: {device}\n")

# ==========================================
# 1. 초고속 데이터 압축 해제 (Drive -> Colab Local)
# ==========================================
drive_zip_path = '/content/drive/MyDrive/Colab_Data/cifar10.tgz'
local_extract_path = '/content/'
local_data_path = '/content/cifar10' # fast.ai 압축을 풀면 나오는 기본 폴더명

if not os.path.exists(local_data_path):
    print("📦 구글 드라이브에서 코랩 로컬로 데이터를 푸는 중... (약 10~15초 소요)")
    with tarfile.open(drive_zip_path, 'r:gz') as tar:
        tar.extractall(path=local_extract_path)
    print("✅ 로컬 압축 해제 완료!\n")
else:
    print("✨ 이미 코랩 로컬에 데이터가 압축 해제되어 있습니다.\n")

# ==========================================
# 2. 데이터셋 및 데이터로더 설정 (Data Augmentation)
# ==========================================
# CIFAR-10 이미지(32x32)에 맞는 기본적인 변환 적용
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# 폴더 구조(train/test)를 읽어오는 ImageFolder 사용
train_dataset = datasets.ImageFolder(root=os.path.join(local_data_path, 'train'), transform=transform_train)
test_dataset = datasets.ImageFolder(root=os.path.join(local_data_path, 'test'), transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"📊 학습 데이터: {len(train_dataset)}장 | 테스트 데이터: {len(test_dataset)}장 준비 완료\n")



🖥️ 현재 사용 중인 장치: cuda

📦 구글 드라이브에서 코랩 로컬로 데이터를 푸는 중... (약 10~15초 소요)


/tmp/ipykernel_466/2139954131.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=local_extract_path)


✅ 로컬 압축 해제 완료!

📊 학습 데이터: 50000장 | 테스트 데이터: 10000장 준비 완료



In [ ]:
import math

# ==========================================
# 3. 모델 정의 (ResNet-18 for CIFAR-10)
# ==========================================
# torchvision의 기본 ResNet-18을 로드하되, CIFAR-10(10개 클래스)에 맞게 마지막 레이어 수정
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

print("✅ ResNet-18 모델 로드 및 CIFAR-10 커스텀 완료!\n")

✅ ResNet-18 모델 로드 및 CIFAR-10 커스텀 완료!



In [ ]:


# ==========================================
# 4. Muon 옵티마이저 핵심 로직 구현
# ==========================================
@torch.no_grad()
def zeropower_via_newtonschulz5(G, steps=5, eps=1e-7):
    """Newton-Schulz 반복을 통한 Gradient 직교화 (Muon의 핵심 연산)"""
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750,  2.0315)
    X = G.bfloat16()
    X /= (X.norm() + eps) # 정규화
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X.to(G.dtype)

class Muon(optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            for p in group['params']:
                g = p.grad
                if g is None:
                    continue

                state = self.state[p]
                if 'momentum_buffer' not in state:
                    state['momentum_buffer'] = torch.zeros_like(g)
                buf = state['momentum_buffer']

                # 2D 이상의 텐서를 2D로 변환 (Conv2d 처리용)
                original_shape = g.shape
                if g.ndim > 2:
                    g = g.view(g.size(0), -1)

                # Newton-Schulz 직교화
                g_ortho = zeropower_via_newtonschulz5(g, steps=group['ns_steps'])
                g_ortho = g_ortho.view(original_shape)

                # Nesterov Momentum 업데이트
                buf.mul_(group['momentum']).add_(g_ortho)
                if group['nesterov']:
                    g_update = g_ortho + group['momentum'] * buf
                else:
                    g_update = buf

                p.add_(g_update, alpha=-group['lr'])
        return loss

# ==========================================
# 5. 파라미터 분리 및 하이브리드 옵티마이저 설정
# ==========================================
muon_params = []
adamw_params = []

for name, param in model.named_parameters():
    # 2D 이상 행렬(Conv, Linear의 weight)은 Muon 적용
    if param.ndim >= 2:
        muon_params.append(param)
    # 1D 파라미터(Bias, BatchNorm 등)는 AdamW 적용
    else:
        adamw_params.append(param)

optimizer_muon = Muon(muon_params, lr=0.02, momentum=0.95)
optimizer_adamw = optim.AdamW(adamw_params, lr=3e-4, weight_decay=0.01)

criterion = nn.CrossEntropyLoss()

print(f"⚙️ 옵티마이저 설정 완료: Muon({len(muon_params)} params) + AdamW({len(adamw_params)} params)\n")


⚙️ 옵티마이저 설정 완료: Muon(21 params) + AdamW(41 params)



In [ ]:
# ==========================================
# 7. SAM 옵티마이저 구현 (Ascent Step + Descent Step)
# ==========================================
class SAM(optim.Optimizer):
    """
    first_step : w -> w + rho * grad/||grad||   (Ascent, "가장 나쁜 방향"으로 임시 이동)
    second_step: w 복원 후 base_optimizer로 실제 업데이트 (Descent)
    """
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        assert rho >= 0
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group['rho'] / (grad_norm + 1e-12)
            for p in group['params']:
                if p.grad is None:
                    continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w)
                self.state[p]['e_w'] = e_w
        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None or 'e_w' not in self.state[p]:
                    continue
                p.sub_(self.state[p]['e_w'])
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()

    def _grad_norm(self):
        shared_device = self.param_groups[0]['params'][0].device
        return torch.norm(torch.stack([
            p.grad.norm(p=2).to(shared_device)
            for group in self.param_groups for p in group['params']
            if p.grad is not None
        ]), p=2)

    def zero_grad(self, set_to_none=True):
        self.base_optimizer.zero_grad(set_to_none=set_to_none)

print("✅ SAM 옵티마이저 정의 완료!\n")

✅ SAM 옵티마이저 정의 완료!



In [ ]:
# ==========================================
# 8. 2단계 트리거 컨트롤러 (Coarse → Fine)
# ==========================================
import collections

class DynamicSwitchController:
    """
    Phase 1→2 (Coarse Trigger): Train Loss EMA가 임계값 이하 + 최근 N에폭 변동폭(std)이 조밀해지면 전환
    Phase 2→3 (Fine Trigger)  : SAM Ascent로 측정한 Sharpness가 최근 M회 동안 안정화(변화율↓)되면 전환

    ⚠️ coarse_loss_threshold, coarse_std_threshold 값은 Muon 베이스라인의
       실제 Train Loss 곡선을 보고 조정해야 함 (지금은 임시값)
    """
    def __init__(self, coarse_loss_threshold=0.5, coarse_window=3, coarse_std_threshold=0.03,
                 fine_window=3, fine_change_threshold=0.02, ema_alpha=0.3):
        self.phase = 1
        self.ema_alpha = ema_alpha
        self.loss_ema = None
        self.loss_window = collections.deque(maxlen=coarse_window)
        self.coarse_loss_threshold = coarse_loss_threshold
        self.coarse_std_threshold = coarse_std_threshold
        self.sharpness_window = collections.deque(maxlen=fine_window)
        self.fine_change_threshold = fine_change_threshold

    def update_coarse(self, epoch_train_loss):
        self.loss_ema = epoch_train_loss if self.loss_ema is None \
            else self.ema_alpha * epoch_train_loss + (1 - self.ema_alpha) * self.loss_ema
        self.loss_window.append(epoch_train_loss)

        if self.phase == 1 and len(self.loss_window) == self.loss_window.maxlen:
            loss_std = torch.tensor(list(self.loss_window)).std().item()
            if self.loss_ema <= self.coarse_loss_threshold and loss_std <= self.coarse_std_threshold:
                self.phase = 2
                print(f"\n🔍 [Phase 1→2] Loss EMA={self.loss_ema:.4f}, 변동폭(std)={loss_std:.4f} → SAM 센서 가동 시작\n")
        return self.phase

    def update_fine(self, sharpness):
        self.sharpness_window.append(sharpness)
        if self.phase == 2 and len(self.sharpness_window) == self.sharpness_window.maxlen:
            vals = list(self.sharpness_window)
            spread = max(vals) - min(vals)
            if spread <= self.fine_change_threshold:
                self.phase = 3
                vals_str = [f'{v:.4f}' for v in vals]
                print(f"\n🎯 [Phase 2→3] Sharpness 안정화 (최근값 {vals_str}) → Muon 종료, SAM 완전 전환\n")
        return self.phase


def measure_sharpness(model, criterion, data_loader, device, rho=0.05, n_batches=3):
    """여러 배치 평균으로 노이즈 완화"""
    model.eval()
    diffs = []
    it = iter(data_loader)
    for _ in range(n_batches):
        inputs, targets = next(it)
        inputs, targets = inputs.to(device), targets.to(device)

        with torch.no_grad():
            loss_w = criterion(model(inputs), targets).item()

        model.zero_grad()
        loss = criterion(model(inputs), targets)
        loss.backward()

        grad_norm = torch.norm(torch.stack([
            p.grad.norm(p=2) for p in model.parameters() if p.grad is not None
        ]), p=2)
        scale = rho / (grad_norm + 1e-12)

        e_ws = []
        with torch.no_grad():
            for p in model.parameters():
                if p.grad is None:
                    continue
                e_w = p.grad * scale
                p.add_(e_w)
                e_ws.append((p, e_w))

        with torch.no_grad():
            loss_w_eps = criterion(model(inputs), targets).item()
            for p, e_w in e_ws:
                p.sub_(e_w)

        model.zero_grad()
        diffs.append(loss_w_eps - loss_w)

    model.train()
    return sum(diffs) / len(diffs)

print("✅ 트리거 컨트롤러 + Sharpness 측정 함수 정의 완료!\n")

✅ 트리거 컨트롤러 + Sharpness 측정 함수 정의 완료!



In [ ]:
# ==========================================
# 9. 하이브리드 학습 루프 (Muon → SAM 동적 스위칭)
# ==========================================
# 공정 비교를 위해 새 모델로 시작 권장 (아래 두 줄 주석 해제)
# model = models.resnet18(weights=None)
# model.fc = nn.Linear(model.fc.in_features, 10); model = model.to(device)

muon_params, adamw_params = [], []
for name, param in model.named_parameters():
    (muon_params if param.ndim >= 2 else adamw_params).append(param)

optimizer_muon = Muon(muon_params, lr=0.02, momentum=0.95)
optimizer_adamw = optim.AdamW(adamw_params, lr=3e-4, weight_decay=0.01)
optimizer_sam = SAM(model.parameters(), optim.SGD, rho=0.05, lr=0.01, momentum=0.9)  # SGD 베이스라인과 동일 lr 계열로 맞춤 필요시 조정

controller = DynamicSwitchController(
    coarse_loss_threshold=0.5, coarse_window=3, coarse_std_threshold=0.03,
    fine_window=3, fine_change_threshold=0.18,
)

criterion = nn.CrossEntropyLoss()
epochs = 50

print("🚀 CIFAR-10 학습 시작 (Muon → SAM 동적 스위칭 하이브리드)...\n")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    start_time = time.time()
    phase = controller.phase
    sharpness_val = None

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        if phase in (1, 2):
            optimizer_muon.zero_grad()
            optimizer_adamw.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer_muon.step()
            optimizer_adamw.step()

            if phase == 2 and sharpness_val is None:
              sharpness_val = measure_sharpness(model, criterion, train_loader, device, rho=0.05, n_batches=3)
        else:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer_sam.first_step(zero_grad=True)

            criterion(model(inputs), targets).backward()
            optimizer_sam.second_step(zero_grad=True)

        running_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    epoch_time = time.time() - start_time
    train_loss = running_loss / len(train_loader)
    test_acc = 100. * correct / total

    controller.update_coarse(train_loss)
    if phase == 2 and sharpness_val is not None:
        controller.update_fine(sharpness_val)

    phase_tag = {1: "P1-Muon", 2: "P2-Muon+Sensor", 3: "P3-SAM"}[phase]
    sharp_str = f" | Sharpness: {sharpness_val:.4f}" if sharpness_val is not None else ""
    print(f"[{phase_tag}] Epoch [{epoch+1}/{epochs}] | Time: {epoch_time:.1f}s | Train Loss: {train_loss:.4f} | Test Acc: {test_acc:.2f}%{sharp_str}")

print("\n🎉 하이브리드 학습 종료!")

🚀 CIFAR-10 학습 시작 (Muon → SAM 동적 스위칭 하이브리드)...

[P1-Muon] Epoch [1/50] | Time: 52.6s | Train Loss: 1.4601 | Test Acc: 62.48%
[P1-Muon] Epoch [2/50] | Time: 48.9s | Train Loss: 1.0068 | Test Acc: 71.49%
[P1-Muon] Epoch [3/50] | Time: 51.0s | Train Loss: 0.8686 | Test Acc: 75.22%
[P1-Muon] Epoch [4/50] | Time: 48.9s | Train Loss: 0.7759 | Test Acc: 75.49%
[P1-Muon] Epoch [5/50] | Time: 49.1s | Train Loss: 0.7431 | Test Acc: 77.55%
[P1-Muon] Epoch [6/50] | Time: 50.6s | Train Loss: 0.6995 | Test Acc: 79.22%
[P1-Muon] Epoch [7/50] | Time: 48.8s | Train Loss: 0.6532 | Test Acc: 79.68%
[P1-Muon] Epoch [8/50] | Time: 49.3s | Train Loss: 0.6291 | Test Acc: 80.15%
[P1-Muon] Epoch [9/50] | Time: 50.0s | Train Loss: 0.5961 | Test Acc: 80.73%
[P1-Muon] Epoch [10/50] | Time: 49.3s | Train Loss: 0.5784 | Test Acc: 81.90%
[P1-Muon] Epoch [11/50] | Time: 50.8s | Train Loss: 0.5607 | Test Acc: 81.35%
[P1-Muon] Epoch [12/50] | Time: 49.3s | Train Loss: 0.5388 | Test Acc: 81.76%
[P1-Muon] Epoch [13/50] | 

In [ ]:
## 📝 실험 기록 — Muon→SAM 하이브리드 v1 (50 epoch, 2026-XX-XX)

### 결과 요약
- Phase 1→2 전환: Epoch 18 (Loss EMA 0.4972, std 0.0100)
- Phase 2→3 전환: Epoch 21 (Sharpness 안정화: 0.39 → 0.27 → 0.32)
- 최종 Test Acc: 84.43% → **SAM 베이스라인(94.69%), AdamW(92.93%)보다 낮음**
- 트리거 감지 로직 자체(Coarse/Fine)는 의도대로 정상 작동함

### ⚠️ 발견된 문제점

**1. 전환 직후 Train Loss 급등 (Epoch 23: 0.44 → 1.64)**
Muon(momentum=0.95)이 21 에폭간 쌓아온 모멘텀 방향 정보가, SGD 기반 SAM으로
전환되는 순간 완전히 리셋됨. 옵티마이저를 교체하면 파라미터 값은 이어지지만
**옵티마이저의 내부 상태(모멘텀 버퍼 등)는 이어지지 않는다**는 걸 실측으로 확인.

**2. 동일 패턴이 Epoch 33에서 재발 (Test Acc 84.5% → 56.4%)**
1회성 전환 충격이라면 몇 에폭 안에 진정되어야 하는데, 10 에폭이나 지나
다시 크게 흔들림. → rho=0.05 + lr=0.01 고정 조합이 "이미 수렴된 좁고
평평한 지형"에서는 구조적으로 과도한 스텝일 가능성. 즉 SAM 하이퍼파라미터를
Muon 베이스라인 초기 설정 그대로 재사용한 게 원인으로 추정됨.

**3. 결과적으로 SAM의 "일반화 성능 개선" 효과가 관측되지 않음**
SAM 베이스라인(문서2)에서는 후반부(40~50 epoch)에 Test Acc가 꾸준히
92%→94%대로 상승하는 패턴을 보였는데, 하이브리드에서는 전환 이후
오히려 정체/진동함. 이는 SAM이 "안정적인 상태에서 미세 조정"을 할 때
효과를 내는데, 지금처럼 매 스파이크마다 크게 흔들리는 상태에서는
그 효과가 가려지는 것으로 보임.

### 🔧 다음 실험에서 시도할 것
- [ ] rho: 0.05 → 0.02~0.03으로 완화 (전환 시점 지형에 맞게)
- [ ] Gradient Clipping 추가 (max_norm=1.0)
- [ ] Phase 3 진입 후 SGD lr에 CosineAnnealing 스케줄러 적용
- [ ] (선택) SGD momentum buffer를 Muon 마지막 상태 기반으로 warm-start 하는 방법 검토
- [ ] rho/lr 조합 자체를 grid search로 짧게 스윕해서 "전환 직후 안정성" 확인

### 💡 계획서(관련 연구) 반영 포인트
"단순 옵티마이저 교체는 파라미터 연속성은 보장되지만 옵티마이저 내부 상태
(모멘텀 등)의 불연속성 문제가 있으며, 이것이 스위칭 직후 학습 불안정성의
주요 원인이 될 수 있다"는 점을 **본 연구의 한계 및 후속 연구 방향**으로
명시할 수 있는 실증적 근거로 활용 가능.

In [ ]:
# ==========================================
# 11. 실험 결과 + 모델 저장 (MyDrive)
# ==========================================
import json
from datetime import datetime

save_dir = '/content/drive/MyDrive/Colab_Data/hybrid_experiments'
os.makedirs(save_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_name = f'muon_sam_hybrid_{timestamp}'

# (1) 하이퍼파라미터 + 전환 시점 요약 (실험 재현/비교용)
run_config = {
    'run_name': run_name,
    'epochs': epochs,
    'muon_lr': 0.02, 'muon_momentum': 0.95,
    'adamw_lr': 3e-4, 'adamw_weight_decay': 0.01,
    'sam_rho': 0.05, 'sam_lr': 0.01, 'sam_momentum': 0.9,
    'coarse_loss_threshold': controller.coarse_loss_threshold,
    'coarse_std_threshold': controller.coarse_std_threshold,
    'fine_change_threshold': controller.fine_change_threshold,
    'final_phase_reached': controller.phase,
    'final_test_acc': test_acc,
    'known_issues': [
        'Phase2->3 전환 직후(epoch23) Train Loss 급등 (0.44->1.64)',
        'SGD momentum buffer가 0으로 리셋되어 관성 정보 손실',
        'rho=0.05가 이미 수렴된 지형 기준으로 과도하게 공격적',
        'epoch33에서 동일 패턴 재발 (Test Acc 84.5%->56.4%), 일시적 충격이 아닌 구조적 불안정 의심',
    ],
}

with open(os.path.join(save_dir, f'{run_name}_config.json'), 'w', encoding='utf-8') as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

# (2) 학습된 모델 가중치
torch.save(model.state_dict(), os.path.join(save_dir, f'{run_name}_model.pt'))

print(f"✅ 저장 완료: {save_dir}/{run_name}_*")

✅ 저장 완료: /content/drive/MyDrive/Colab_Data/hybrid_experiments/muon_sam_hybrid_20260717_155101_*


In [ ]:
# ==========================================
# 10. 하이브리드 학습 루프 v2 (rho 완화 + Gradient Clipping 적용)
# ==========================================
# 반드시 새 모델로 시작 (v1과 공정 비교를 위해 주석 해제 필수)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

muon_params, adamw_params = [], []
for name, param in model.named_parameters():
    (muon_params if param.ndim >= 2 else adamw_params).append(param)

optimizer_muon = Muon(muon_params, lr=0.02, momentum=0.95)
optimizer_adamw = optim.AdamW(adamw_params, lr=3e-4, weight_decay=0.01)
optimizer_sam = SAM(model.parameters(), optim.SGD, rho=0.02, lr=0.01, momentum=0.9)  # rho: 0.05→0.02

controller = DynamicSwitchController(
    coarse_loss_threshold=0.5, coarse_window=3, coarse_std_threshold=0.03,
    fine_window=3, fine_change_threshold=0.18,
)

criterion = nn.CrossEntropyLoss()
epochs = 50

print("🚀 CIFAR-10 학습 시작 (Muon → SAM 하이브리드 v2: rho↓ + grad clip)...\n")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    start_time = time.time()
    phase = controller.phase
    sharpness_val = None

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        if phase in (1, 2):
            optimizer_muon.zero_grad()
            optimizer_adamw.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer_muon.step()
            optimizer_adamw.step()

            if phase == 2 and sharpness_val is None:
                sharpness_val = measure_sharpness(model, criterion, train_loader, device, rho=0.05, n_batches=3)
        else:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer_sam.first_step(zero_grad=True)

            criterion(model(inputs), targets).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ← 추가
            optimizer_sam.second_step(zero_grad=True)

        running_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    epoch_time = time.time() - start_time
    train_loss = running_loss / len(train_loader)
    test_acc = 100. * correct / total

    controller.update_coarse(train_loss)
    if phase == 2 and sharpness_val is not None:
        controller.update_fine(sharpness_val)

    phase_tag = {1: "P1-Muon", 2: "P2-Muon+Sensor", 3: "P3-SAM"}[phase]
    sharp_str = f" | Sharpness: {sharpness_val:.4f}" if sharpness_val is not None else ""
    print(f"[{phase_tag}] Epoch [{epoch+1}/{epochs}] | Time: {epoch_time:.1f}s | Train Loss: {train_loss:.4f} | Test Acc: {test_acc:.2f}%{sharp_str}")

print("\n🎉 하이브리드 v2 학습 종료!")

🚀 CIFAR-10 학습 시작 (Muon → SAM 하이브리드 v2: rho↓ + grad clip)...

[P1-Muon] Epoch [1/50] | Time: 51.3s | Train Loss: 1.4489 | Test Acc: 64.30%
[P1-Muon] Epoch [2/50] | Time: 49.5s | Train Loss: 1.0781 | Test Acc: 71.91%
[P1-Muon] Epoch [3/50] | Time: 49.0s | Train Loss: 0.9384 | Test Acc: 71.29%
[P1-Muon] Epoch [4/50] | Time: 50.8s | Train Loss: 0.8704 | Test Acc: 75.53%
[P1-Muon] Epoch [5/50] | Time: 49.1s | Train Loss: 0.8252 | Test Acc: 75.89%
[P1-Muon] Epoch [6/50] | Time: 50.3s | Train Loss: 0.7781 | Test Acc: 77.15%
[P1-Muon] Epoch [7/50] | Time: 49.8s | Train Loss: 0.7191 | Test Acc: 77.31%
[P1-Muon] Epoch [8/50] | Time: 49.1s | Train Loss: 0.6741 | Test Acc: 79.88%
[P1-Muon] Epoch [9/50] | Time: 51.1s | Train Loss: 0.6656 | Test Acc: 80.11%
[P1-Muon] Epoch [10/50] | Time: 49.4s | Train Loss: 0.6406 | Test Acc: 81.88%
[P1-Muon] Epoch [11/50] | Time: 50.0s | Train Loss: 0.6181 | Test Acc: 81.64%
[P1-Muon] Epoch [12/50] | Time: 49.8s | Train Loss: 0.6009 | Test Acc: 81.49%
[P1-Muon] Ep